# 卫星姿态控制 AI 部分（强化学习）—— 项目逻辑与代码说明

本 Notebook 面向项目组成员，细致说明 `satellite_ai` 文件夹里 **强化学习（AI）部分** 在做什么、背后的逻辑，以及每一部分对应哪些代码。

- 所有改进点均以 `ds:` 编号标注，对应关系见 `README.md` 的清单表。
- 环境：`sssiiieee的文件夹/venv`（Python 3.14 + torch 2.14 + stable-baselines3 2.9 + gymnasium 1.3）。


## 一、AI 部分在解决什么问题

**背景**：卫星姿态控制的目标是让卫星从某个姿态回到目标姿态（本项目里是归零），本质是求一个「控制力矩 → 姿态」的规律。

**传统方法（PID）的局限**：
- 需要较精确的动力学模型 + 人工整定参数；
- 面对三轴耦合、非线性、执行器饱和等约束时，鲁棒性与自适应能力不足。

**强化学习（RL）的思路**：
- 不显式推导控制律，而是把控制问题改造成一个「试错学习」问题；
- 让智能体（Agent）与环境反复交互，自己学出一个「从状态到动作」的映射（即**策略 policy**）。

**本项目 AI 部分 = 两件事**：
1. **环境（Environment）**：把卫星姿态动力学包装成标准的 Gym 环境 —— 对应 `sat_env.py`；
2. **智能体（Agent）**：用 PPO / SAC 算法训练出控制策略 —— 对应 `train.py`；
3. 配套的评估与泛化测试 —— 对应 `metrics.py` 与 `evaluate.py`。


## 二、强化学习的基本框架

RL 的核心是这样一个**闭环交互**：

```
            action（动作：控制力矩）
   Agent ──────────────────────────────► Environment
   （策略）                              （卫星动力学）
        ◄──────────────────────────────
         state（状态）+ reward（奖励）
```

**关键概念**：

| 概念 | 本项目中的含义 | 代码位置 |
|------|---------------|----------|
| state 状态 | 归一化的角度 + 角速度 `[θ_norm, ω_norm]` | `sat_env.py::_get_obs` |
| action 动作 | 归一化控制力矩 ∈ [-1, 1] | `sat_env.py::step` |
| reward 奖励 | 越小越差的代价函数（负值） | `sat_env.py::_compute_reward` |
| episode 回合 | 从初始姿态到收敛/超时/发散的整个过程 | `sat_env.py::reset` + `step` |
| policy 策略 | 神经网络，输入状态输出动作 | `train.py::build_model` |

训练的过程，就是**不断调策略网络参数，让长期累计奖励（累计代价的负值）最大化**。


In [ ]:
# 先确认环境：导入本文件夹的模块 + 打印关键库版本
import sys, os
sys.path.insert(0, os.getcwd())   # 让本目录下的 sat_env / metrics 能被 import

import numpy as np
import gymnasium as gym
import stable_baselines3 as sb3

from single_axis.sat_env import SatelliteAttitudeEnv
import common.metrics as metrics

print("gymnasium", gym.__version__, "| stable_baselines3", sb3.__version__)
print("import OK")

## 三、环境：把卫星问题包装成 Gym 环境（sat_env.py）

强化学习算法（PPO/SAC）不关心具体物理，只认标准的 `reset()/step()` 接口。
所以第一步是把卫星动力学包装成 `gym.Env`。环境由三要素组成：

1. **状态空间 state**：智能体能看到什么；
2. **动作空间 action**：智能体能输出什么；
3. **奖励函数 reward**：智能体做得好不好。

动力学模型是最简化的**单轴刚体（双积分器）**：

$$\ddot{\theta} = \frac{\tau}{I}$$

用欧拉积分一步步推进：`omega += alpha*dt`，`theta += omega*dt`。


### 3.1 动力学模型 —— `Satellite` 类

对应 `sat_env.py` 中的 `class Satellite`：

```python
class Satellite:
    def apply_torque(self, torque, dt):
        alpha = torque / self.I          # 角加速度 = 力矩 / 转动惯量
        self.omega += alpha * dt         # 更新角速度
        self.theta += self.omega * dt    # 更新角度
```

这里 `I` 是转动惯量，`theta` 是角度(rad)，`omega` 是角速度(rad/s)。


### 3.2 状态 state —— 归一化观测（ds5）

智能体只能看到「观测」，本项目里观测就是归一化后的 `[角度, 角速度]`：

```python
def _get_obs(self):
    theta_norm = self.sat.theta / self.config['theta_scale']   # / 90°
    omega_norm = self.sat.omega / self.config['omega_scale']   # / 60°/s
    return np.array([theta_norm, omega_norm], dtype=np.float32)
```

- **ds5 改进**：统一把观测缩放到 `[-1, 1]`，避免角度/角速度量纲差异导致神经网络训练不稳定。
- 注意：观测是**归一化**的，不能直接用于物理指标；所以 `step()` 的 `info` 里额外保留了真实的 `theta/omega/torque`（见 3.5）。


### 3.3 动作 action —— 归一化力矩

智能体输出的是归一化动作 `a ∈ [-1, 1]`，环境内部再映射成真实力矩：

```python
torque = self.config['max_torque'] * np.tanh(float(action[0]))
```

- 用 `tanh` 而不是硬 `clip`：tanh 是光滑函数，梯度更友好，训练更稳定。


### 3.4 奖励函数（核心，ds2）

奖励是 RL 的「指挥棒」——智能体唯一的目标就是最大化累计奖励。
对应 `sat_env.py::_compute_reward`，设计成**负的二次代价**：

```
reward = -( w_theta·e² + w_omega·ω² + w_torque·τ² ) - time_penalty
```

- 第一项惩罚角度误差 `e`（离目标越远扣越多）；
- 第二项惩罚角速度（相当于「阻尼」，防止震荡）；
- 第三项惩罚能耗（力矩越大越费能量）；
- 最后 `time_penalty` 是**每步时间惩罚**，鼓励「又快又稳」，避免策略磨蹭。

此外在 `step()` 里还有两个**离散奖励**：

| 事件 | 奖励 | 目的 |
|------|------|------|
| 成功收敛（持续稳定 10 步） | `+success_bonus` | 给「到达目标」一个明确的正向信号 |
| 发散（角度 > 90°） | `-diverge_penalty` | 惩罚失稳 |

- **ds2 改进**：去掉了旧版手工调的交叉项 `-2·θ·ω`（该交叉项在不同象限会变相奖励/惩罚，权重偏大易诱发振荡），改成清晰可解释的三项二次代价 + 成功奖励 + 时间惩罚。


In [ ]:
# 直接打印奖励函数的源码，对照上面的讲解
import inspect
print(inspect.getsource(SatelliteAttitudeEnv._compute_reward))

### 3.5 终止条件（ds1，一个关键修正）

一个回合（episode）在三种情况下结束，Gym 用两个布尔标志区分：

| 结束原因 | 标志 | 含义 |
|----------|------|------|
| 成功收敛 / 发散 | `terminated = True` | 「这个状态已经是终态，后面没有未来价值」 |
| 步数超时 | `truncated = True` | 「被截断，但状态本身还有未来」 |

- **ds1 改进**：旧代码把「超时」也设成了 `terminated`。这会让算法误以为超时的状态没有未来价值，在时间截断处**错误地截断价值函数的 bootstrap**，拖慢收敛甚至导致发散。正确做法是超时用 `truncated`。


### 3.6 干扰（ds3）与域随机化（ds4）

**干扰**：真实卫星会受到环境力矩，旧版只有幅值 0.005 的极弱白噪声。改进后（`step()` 内）：

```
disturbance = 偏置 + 周期正弦项·sin(2πft) + 高斯噪声
```

- **ds3 改进**：更贴近「常值偏置 + 轨道周期扰动 + 随机噪声」的真实组合。

**域随机化**：训练时每次 `reset()` 随机抽取惯量 `I`、初始角度/角速度、干扰幅值：

```python
self.sat.I = self.np_random.uniform(*cfg['I_range'])   # 0.8 ~ 1.2
```

- **ds4 改进**：让策略在「各种参数下」都能控好，而不是只记住某一组参数，增强**泛化能力**。


In [ ]:
# 动手：创建环境，用随机动作跑几步，观察状态 / 奖励 / 终止标志
env = SatelliteAttitudeEnv(config={'disturbance': True, 'randomize': True})
obs, _ = env.reset(seed=0)
print("初始观测（归一化）:", obs)
print("动作空间:", env.action_space)
print("观测空间:", env.observation_space)

for i in range(3):
    action = env.action_space.sample()          # 随机动作（还没训练，纯演示）
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step {i}: theta={np.degrees(info['theta']):6.2f} deg  "
          f"omega={np.degrees(info['omega']):6.2f} deg/s  "
          f"reward={reward:7.3f}  terminated={terminated}  truncated={truncated}")

## 四、训练：让智能体学会控制（train.py）

训练脚本 `train.py` 的流程：

```
建环境(make_env) → 包装 VecNormalize → 建模型(SAC/PPO) → model.learn() → 保存
```

对应代码位置：`train.py::make_env`、`build_model` 和主流程。


### 4.1 算法：SAC 与 PPO（ds7）

| 算法 | 特点 | 本项目定位 |
|------|------|-----------|
| **SAC** | 连续控制首选，最大熵、样本效率高、对超参不敏感 | **主力**（`train.py sac` 默认） |
| PPO | 简单稳定、on-policy | 对比 |

- **ds7 改进**：新增 SAC，呼应中期报告「探索 SAC/TD3」的计划。


### 4.2 随机种子（ds6）与奖励归一化（ds9）

- **ds6**：`set_random_seed(SEED)` 固定随机种子，保证实验可复现。
- **ds9**：用 `VecNormalize(norm_obs=False, norm_reward=True)` 归一化**奖励**，让价值函数学习更稳定。
  - 观测已经在环境里归一化过（ds5），所以 `norm_obs=False`。


### 4.3 训练/评估环境分离 + EvalCallback（ds8）

- **训练环境**：开干扰 + 域随机化，逼策略学得更鲁棒；
- **评估环境**：确定性、不加干扰，保证评估可复现、可对比；
- **EvalCallback**：每隔若干步在评估环境上打分，并**自动保存最优模型**到 `satellite_ai_best_model/`。


## 五、评估：怎么判断「学得好」（metrics.py + evaluate.py）

旧项目里指标定义散落且不一致（settling time 两处定义不同、overshoot 会误算），
导致结论难以复现。因此新增了统一评估模块 `metrics.py`（**ds10**）。


### 5.1 统一指标

| 指标 | 含义 |
|------|------|
| settling_time | 稳定时间：进入 ±band 且之后**持续稳定**的时刻 |
| overshoot | 超调量：越过目标到另一侧的最大偏离（度 / 百分比） |
| steady_error | 稳态误差：末端平均绝对误差 |
| IAE / ITAE / ISE | 误差积分指标（衡量整体跟踪质量） |
| control_energy | 控制能耗 ∫τ²dt |
| success | 是否收敛到目标带内 |


In [ ]:
# 演示统一指标：构造一条指数收敛的轨迹
t = np.arange(0, 10.0, 0.02)
theta = 30.0 * np.exp(-t)      # 30° 指数收敛到 0°
torque = np.zeros_like(t)

m = metrics.compute_metrics(t, theta, torque=torque)
metrics.print_metrics(m, name="示例轨迹（指数收敛）")

### 5.2 OOD 泛化测试（ds11）

`evaluate.py` 在**训练范围之外**的初始条件下测试策略（例如 90° 大角度、非零初始角速度），
并统一输出量化指标表，而不是只画几条曲线。这能回答「策略是不是真的会控制，还是只记住了训练样本」。


In [ ]:
# 可选：加载训练好的模型并评估（需先运行 train.py 生成模型，否则提示跳过）
from stable_baselines3 import SAC, PPO

model_path = "single_axis/output/models/sac_satellite_final"
if os.path.exists(model_path + ".zip"):
    model = SAC.load(model_path)
    env3 = SatelliteAttitudeEnv(config={'disturbance': False, 'randomize': False})
    env3.reset(seed=0)
    env3.sat.set_state(np.radians(30.0), 0.0)
    env3.step_count = 0
    obs = env3._get_obs()

    ts, ths, tqs = [], [], []
    for i in range(env3.config['max_steps']):
        ts.append(i * env3.config['dt'])
        ths.append(np.degrees(env3.sat.theta))
        a, _ = model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env3.step(a)
        tqs.append(info['torque'])
        if term or trunc:
            break

    m = metrics.compute_metrics(np.array(ts), np.array(ths), torque=np.array(tqs))
    metrics.print_metrics(m, name="SAC 控制器（30° 初始）")
else:
    print("未找到模型，请先在命令行运行：../venv/Scripts/python.exe single_axis/train.py sac")

## 六、代码结构与 ds 改进点总览

**文件 → 作用**

| 文件 | 作用 |
|------|------|
| `sat_env.py` | 卫星姿态 Gym 环境（状态/动作/奖励/动力学/终止/干扰/域随机化） |
| `train.py` | 训练脚本（SAC/PPO、种子、奖励归一化、评估回调） |
| `metrics.py` | 统一评估函数 |
| `evaluate.py` | 训练后评估 + OOD 泛化测试 |

**ds 编号 → 改进内容**

| 编号 | 改进 |
|------|------|
| ds1 | 终止语义修正（超时用 truncated） |
| ds2 | 奖励函数系统化（成功奖励 + 时间惩罚） |
| ds3 | 干扰模型真实化（偏置 + 周期 + 噪声） |
| ds4 | 域随机化（随机惯量/初始状态/干扰） |
| ds5 | 观测归一化统一 + info 保留真实物理量 |
| ds6 | 固定随机种子 |
| ds7 | 新增 SAC 算法 |
| ds8 | 训练/评估分离 + EvalCallback |
| ds9 | VecNormalize 归一化奖励 |
| ds10 | 统一评估指标模块 |
| ds11 | OOD 泛化评估 |


## 七、小结与下一步

**一句话总结 AI 部分**：
> 把「卫星姿态控制」包装成强化学习问题（`sat_env.py`），用 SAC/PPO 训练出一个
> 「输入姿态状态、输出控制力矩」的神经网络策略（`train.py`），再用统一指标和
> 分布外测试评估它（`metrics.py` / `evaluate.py`）。

**尚未实现（下一步方向）**：
1. 三轴姿态控制环境（复用 `satellite1PID/week5` 的刚体模型 `satellite3d.py`）；
2. 自动调参：从随机搜索升级为 Optuna 贝叶斯优化；
3. 执行器模型：反作用轮（力矩限幅 + 速率限幅 + 动量饱和）。


## 八、训练成果文件解读

运行 `train.py`（训练）和 `evaluate.py`（评估）后，`satellite_ai` 文件夹里会生成一批成果文件。
本节说明每个文件是什么、以及如何阅读它们。


### 8.1 文件清单

| 文件 / 目录 | 是什么 |
|------------|--------|
| `single_axis/output/models/best_model.zip` | 训练过程中**评估得分最高**的模型（EvalCallback 自动保存） |
| `single_axis/output/models/sac_satellite_final.zip` | 训练结束时的最终模型 |
| `single_axis/output/models/sac_vecnormalize.pkl` | 奖励归一化统计量（用于继续训练 / 精确复现） |
| `single_axis/output/models/eval_logs/evaluations.npz` | 每次评估的得分记录（训练步数 + 每回合奖励 + 回合长度） |
| `single_axis/output/tensorboard/` | TensorBoard 训练曲线（reward / loss / ent_coef 等） |
| `single_axis/output/sac_ood_generalization.png` | 泛化测试的收敛曲线图（`evaluate.py` 生成） |
| `single_axis/output/train_log.txt` | 训练过程的控制台日志 |


### 8.2 如何阅读这些文件

- **模型文件（.zip）**：用 `SAC.load()` / `PPO.load()` 加载，再用 `model.predict(obs)` 得到动作；
  这是唯一「真正可用」的成果。
- **eval_logs/evaluations.npz**：记录每个评估时刻的「平均奖励」与「回合长度」。
  - 平均奖励是**负值**（代价的负数），**越接近 0 越好**；
  - 回合长度越短（< max_steps），说明越早收敛成功。
- **tensorboard**：`python -m tensorboard --logdir satellite_ai_tensorboard`，可交互查看曲线。
- **泛化测试图**：横轴时间、纵轴角度，看不同初始角能否收敛到 0。


In [ ]:
# 读取 EvalCallback 保存的评估记录，画「评估奖励随训练变化」曲线
import numpy as np, matplotlib.pyplot as plt, os

path = "single_axis/output/models/eval_logs/evaluations.npz"
if os.path.exists(path):
    d = np.load(path)
    ts = d["timesteps"]                 # 每次评估时的训练步数
    rs = d["results"].mean(axis=1)      # 每次评估的平均奖励（负值，越接近 0 越好）
    el = d["ep_lengths"].mean(axis=1)   # 平均回合长度（越短越早收敛）

    for t, r, e in zip(ts, rs, el):
        print(f"step {t:>7}: mean_reward={r:8.1f}  mean_ep_len={e:6.1f}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ts, rs, "o-")
    ax[0].set_xlabel("train timesteps"); ax[0].set_ylabel("mean eval reward")
    ax[0].set_title("Eval reward (closer to 0 is better)"); ax[0].grid()
    ax[1].plot(ts, el, "o-", color="orange")
    ax[1].set_xlabel("train timesteps"); ax[1].set_ylabel("mean episode length")
    ax[1].set_title("Episode length (shorter = earlier success)"); ax[1].grid()
    plt.tight_layout(); plt.show()
else:
    print("未找到 eval_logs/evaluations.npz（先运行 single_axis/train.py）")

### 8.3 本次训练的实际结果（示例）

本次用 SAC 训练 10 万步后，`evaluate.py` 输出的泛化指标：

| 初始条件 | 稳定时间 | 超调 | 成功 |
|----------|---------|------|------|
| 10° / 0°/s  | 1.98s | 17.8% | ✅ |
| 30° / 0°/s  | 2.68s | 6.1%  | ✅ |
| 60° / 0°/s  | 3.40s | 3.1%  | ✅ |
| 90° / 0°/s  | 3.94s | 2.0%  | ✅ |
| 30° / 10°/s | 2.82s | 6.1%  | ✅ |

**解读**：
- 所有初始条件（包括 90° 大角度这个**训练范围之外**的情况）都 `success=True`，说明策略具备一定泛化能力；
- 初始角越大，稳定时间越长（合理：转得更远）；
- 评估奖励曲线在训练中会起伏（见上一个单元格输出），这是 RL 训练的正常现象，
  `best_model.zip` 保存的是评估得分最高的那次（本例约为第 9 万步处），
  有时它会比 `sac_satellite_final.zip`（训练末尾）略好。


## 九、三轴强化学习模块（新增）

> 推进记录：二次奖励→10.9°；线性奖励(ds21)→2.7°；
> 课程学习(ds22) + potential-based shaping(ds23) + 100 万步/大网络(ds24)→训练中。

单轴 RL 处理的是 1 自由度双积分器（可解析求解的线性系统），不足以体现 RL 的价值。
三轴姿态控制才是真正有耦合、有非线性的问题，也是本项目「AI 在姿态控制中应用」的核心场景。
本模块从 PID 侧复制了正确的三轴刚体模型（`satellite3d.py`），在其上搭 RL。

### 9.1 三轴动力学

刚体姿态动力学由两部分组成（`satellite3d.py`）：

- **动力学（欧拉方程）**：`I·ω̇ + ω×(Iω) = τ`，其中 `ω×(Iω)` 是**陀螺耦合项**，三轴互相影响；
- **运动学（四元数）**：`q̇ = 0.5·q⊗(0,ω)`，用四元数避免欧拉角的奇异性。

单轴只是它的退化特例（没有耦合项）。


### 9.2 状态 / 动作 / 奖励设计（sat_env3d.py）

| 要素 | 设计 | 说明 |
|------|------|------|
| 状态(7维) | 误差四元数 q_err(4) + 归一化角速度(3) | q_err = q_target ⊗ q_current⁻¹，无奇异性 |
| 动作(3维) | 三轴力矩 `max_torque·tanh(a)` | 每轴 ±2 N·m，与 PID 侧一致 |
| 奖励 | `-(w_angle·e + w_omega·‖ω‖ + w_torque·‖τ‖²) - 时间惩罚` | e = 等效转角 `2·arccos(|q_err_w|)`（ds21 改为线性惩罚） |
| 终止 | 成功/发散 → terminated；超时 → truncated | 沿用 ds1 规范 |

误差用**等效转角**（误差四元数对应的旋转角度）这一个标量来衡量，使三轴问题能复用单轴的 `metrics.py` 评估。


### 9.3 PID vs RL 对比（evaluate3d.py）

`evaluate3d.py` 在**同一动力学、同一初始姿态、同一力矩限幅**下对比 PID（`pid3d.py`，参数取 PID 侧整定结果）与 SAC：

- 单轴小角度：PID 通常就能收敛；
- **三轴同时大角度（如 45°/45°/45°）**：耦合项让 PID 难以整定、容易失稳，这正是 RL 的优势场景。


In [ ]:
# 创建三轴环境跑几步，观察等效转角误差
from three_axis.sat_env3d import Satellite3DEnv
import numpy as np

env = Satellite3DEnv(config={'disturbance': False, 'randomize': False})
obs, _ = env.reset(seed=0)
print("观测维度:", env.observation_space.shape, "| 动作维度:", env.action_space.shape)

for i in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step {i}: angle_err={np.degrees(info['angle_err']):6.2f} deg  "
          f"reward={reward:7.3f}  terminated={terminated}  truncated={truncated}")

### 9.4 对应代码

| 文件 | 作用 |
|------|------|
| `three_axis/sat_env3d.py` | 三轴 Gym 环境（ds14~ds17、ds21~ds23） |
| `three_axis/train3d.py` | 三轴训练脚本（ds18、ds22、ds24） |
| `three_axis/evaluate3d.py` | 三轴评估 + PID vs RL 对比（ds19、ds20） |
| `three_axis/satellite3d.py` / `quaternion_utils.py` / `pid3d.py` | 从 PID 侧复制的动力学/四元数/PID（ds13） |
| `common/metrics.py` | 统一评估函数（单轴/三轴共用，ds10） |


## 十、模型辨识 + MPC 模块（补齐立项第②项）

立项书三大内容：①动力学建模 ②**模型辨识 + 预测控制** ③梳理 DRL 原理。
其中 ② 一直空白，本模块补齐它，让项目「经典控制(PID) / 智能控制(RL) / 预测控制(MPC)」三条路线齐备。

### 10.1 系统辨识（mpc/system_id.py，ds27）

用随机力矩激励卫星，采集 (x_k, u_k, x_{k+1}) 样本，再用**最小二乘（线性回归）**
拟合离散线性状态空间模型 `x_{k+1} = A x_k + B u_k`。单轴是线性双积分器，
最小二乘能精确恢复真实 A、B（误差 2e-16，机器精度）；对非线性系统则需用神经网络辨识。


### 10.2 MPC 控制（mpc/mpc_controller.py，ds28）

MPC = 「预测 + 优化 + 滚动」三步：

1. **预测**：用辨识模型把未来 N 步状态堆叠成 `X = M·x0 + L·U`；
2. **优化**：最小化 `Σ(xᵀQx + R u²)`，这是一个无约束二次规划，闭式解 `U* = -H⁻¹f`；
3. **滚动**：只取第一步控制量 `u0`，下个时刻重新预测优化（receding horizon），力矩做饱和限幅。


### 10.3 三方对比（mpc/compare.py，ds29）

同一动力学/初始条件/力矩限幅下，对比网格整定 PID、训练好的 SAC、基于辨识模型的 MPC：

| 指标（30°） | PID | MPC | SAC | 谁最优 |
|------------|-----|-----|-----|--------|
| 稳定时间 | 2.72s | 1.66s | 2.68s | MPC |
| 超调量 | 15.9% | 3.6% | 6.1% | MPC |
| 稳态误差 | 0.002° | 0.0000° | 0.43° | MPC |
| 控制能耗 | 1.12 | 1.87 | 0.90 | SAC |

结论：MPC 靠精确模型在速度/超调/精度上全面最优（能耗略高，Q/R 可调）；SAC 省能量；
PID 简单但慢、超调大。


In [ ]:
# 演示：系统辨识 + MPC 控制
from mpc.system_id import generate_data, identify
from mpc.mpc_controller import MPCController
from single_axis.sat_env import Satellite
import numpy as np

X, U, Y = generate_data()
A, B = identify(X, U, Y)
print("辨识 A =\n", np.round(A, 4))
print("辨识 B =\n", np.round(B, 4))

mpc = MPCController(A, B, np.diag([100.0, 5.0]), 0.5, horizon=20, max_torque=2.0)
sat = Satellite(I=1.0)
sat.set_state(np.radians(30.0), 0.0)
for i in range(5):
    u = mpc.control(np.array([sat.theta, sat.omega]))
    sat.apply_torque(u, 0.02)
    print(f"step {i}: theta={np.degrees(sat.theta):6.2f} deg, torque={u:6.3f} Nm")

### 10.4 对应代码

| 文件 | 作用 |
|------|------|
| `mpc/system_id.py` | 系统辨识（ds27） |
| `mpc/mpc_controller.py` | MPC 控制器（ds28） |
| `mpc/compare.py` | 三方对比 PID/MPC/SAC（ds29） |


## 十一、三轴姿态可视化动画（ds30）

`visualization/animate_attitude.py` 用 matplotlib 3D 动画直观演示三轴卫星从初始姿态
收敛到目标姿态（单位四元数）的过程：

- **左图（3D）**：画出卫星本体坐标系（红X / 绿Y / 蓝Z）随四元数 `q` 的旋转，灰色虚线是目标零姿态；
- **右图（曲线）**：等效转角误差 `2·arccos(|q_err_w|)` 随时间收敛。

脚本分别用 SAC 与 PID 两个控制器仿真同一条轨迹，生成两个 GIF 便于对比，
直观印证了三轴 RL「单轴能收敛、多轴发散」的结论。
